# 🤖 AI-Generated vs Real Image Classification
### Three Models: MobileNetV2 · EfficientNetB0 · NASNetMobile
---

In [1]:
# ============================================================
# CELL 1 — Imports & Setup
# ============================================================
import os, warnings, logging
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')
logging.getLogger('tensorflow').setLevel(logging.ERROR)

import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend — saves files reliably
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
import matplotlib.patheffects as pe
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, applications
from sklearn.metrics import classification_report, confusion_matrix
import glob, random, time
from pathlib import Path

# Reproducibility
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# Config
IMG_SIZE    = (224, 224)
BATCH_SIZE  = 32
AUTOTUNE    = tf.data.AUTOTUNE
CLASS_NAMES = ['AI-Generated', 'Real']

# ── Output folder — ALL graphs saved here, not in base dir ──
OUTPUT_DIR = Path('classifier_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)
print(f"All outputs → {OUTPUT_DIR.resolve()}")

print(f"TensorFlow  : {tf.__version__}")
print(f"GPU devices : {tf.config.list_physical_devices('GPU') or 'CPU only'}")
print("Setup complete ✓")

All outputs → C:\Users\nikhi\OneDrive\Desktop\Classifier\classifier_outputs
TensorFlow  : 2.21.0
GPU devices : CPU only
Setup complete ✓


In [2]:
# ============================================================
# CELL 2 — Load & Validate Dataset Paths
# ============================================================
from pathlib import Path
import random

# ── Adjust this to point to your archive folder ──────────────
BASE_DIR = Path(r"C:/Users/nikhi/OneDrive/Desktop/Classifier/archive")

AI_DIR   = BASE_DIR / "AiArtData"
REAL_DIR = BASE_DIR / "RealArt"

EXTS = (".jpg", ".jpeg", ".png", ".webp")

# ── STEP 1: Collect all candidate paths ─────────────────────
def collect_paths(folder, label):
    """Return unique (path_str, label) for every supported image in folder."""
    paths = set()
    for ext in EXTS:
        paths.update(str(p) for p in Path(folder).rglob(f"*{ext}"))
        paths.update(str(p) for p in Path(folder).rglob(f"*{ext.upper()}"))
    return [(p, label) for p in paths]

ai_samples   = collect_paths(AI_DIR,   label=0)
real_samples = collect_paths(REAL_DIR, label=1)

print(f"Candidates — AI: {len(ai_samples)}  |  Real: {len(real_samples)}")

# ── STEP 2: Validate images — remove broken ones BEFORE training
print("\nValidating images (this may take a moment)...")

def is_valid_image(path):
    """Try to decode; return False if image is corrupt or unreadable."""
    try:
        raw = tf.io.read_file(path)
        img = tf.image.decode_image(raw, channels=3, expand_animations=False)
        # Must have valid spatial dims
        if img.shape[0] is None or img.shape[1] is None:
            return False
        if img.shape[0] < 10 or img.shape[1] < 10:
            return False
        return True
    except Exception:
        return False

def validate_samples(samples, name):
    good, bad = [], []
    for i, (path, label) in enumerate(samples):
        if i % 500 == 0:
            print(f"  {name}: checked {i}/{len(samples)}...", end='\r')
        if is_valid_image(path):
            good.append((path, label))
        else:
            bad.append(path)
    print(f"  {name}: {len(good)} valid, {len(bad)} removed (corrupt/tiny)   ")
    return good, bad

ai_samples,   ai_bad   = validate_samples(ai_samples,   'AI-Generated')
real_samples, real_bad = validate_samples(real_samples, 'Real')

if ai_bad or real_bad:
    print(f"\n  Removed {len(ai_bad)} AI + {len(real_bad)} Real corrupt images.")

# ── STEP 3: Combine & Split ──────────────────────────────────
all_samples = ai_samples + real_samples
random.shuffle(all_samples)

all_paths  = [s[0] for s in all_samples]
all_labels = [s[1] for s in all_samples]

n_ai, n_real, n_total = len(ai_samples), len(real_samples), len(all_samples)

print(f"\n✅ Final dataset:")
print(f"   AI-Generated : {n_ai}")
print(f"   Real         : {n_real}")
print(f"   Total        : {n_total}")

n_train = int(0.70 * n_total)
n_val   = int(0.15 * n_total)

train_paths  = all_paths[:n_train]
train_labels = all_labels[:n_train]
val_paths    = all_paths[n_train:n_train+n_val]
val_labels   = all_labels[n_train:n_train+n_val]
test_paths   = all_paths[n_train+n_val:]
test_labels  = all_labels[n_train+n_val:]

print(f"\nSplit → Train: {len(train_paths)} | Val: {len(val_paths)} | Test: {len(test_paths)}")

Candidates — AI: 6130  |  Real: 6088

Validating images (this may take a moment)...
  AI-Generated: 6130 valid, 0 removed (corrupt/tiny)   
  Real: 6084 valid, 4 removed (corrupt/tiny)   

  Removed 0 AI + 4 Real corrupt images.

✅ Final dataset:
   AI-Generated : 6130
   Real         : 6084
   Total        : 12214

Split → Train: 8549 | Val: 1832 | Test: 1833


In [3]:
# ============================================================
# CELL 3 — Graphs BEFORE Training  (all saved to OUTPUT_DIR)
# ============================================================

# ── Dark theme ──────────────────────────────────────────────
BG      = '#0D1117'
PANEL   = '#161B22'
BORDER  = '#30363D'
TEXT    = '#E6EDF3'
MUTED   = '#8B949E'
AI_COL  = '#FF6B6B'
REAL_COL= '#3FB950'
ACCENT  = '#58A6FF'
GOLD    = '#F0B429'
PURPLE  = '#BC8CFF'

def apply_dark_style(fig, axes_list):
    fig.patch.set_facecolor(BG)
    for ax in axes_list:
        if ax is None: continue
        ax.set_facecolor(PANEL)
        ax.tick_params(colors=MUTED, labelsize=9)
        ax.xaxis.label.set_color(MUTED)
        ax.yaxis.label.set_color(MUTED)
        ax.title.set_color(TEXT)
        for spine in ax.spines.values():
            spine.set_color(BORDER)
        ax.grid(color=BORDER, linewidth=0.5, linestyle='--', alpha=0.6)

def save_fig(fig, name):
    path = OUTPUT_DIR / name
    fig.savefig(path, dpi=150, bbox_inches='tight', facecolor=BG)
    plt.close(fig)
    print(f"  Saved → {path}")

# ── 3A: Class Distribution + Split ──────────────────────────
fig = plt.figure(figsize=(18, 6), facecolor=BG)
fig.suptitle('Dataset Overview', color=TEXT, fontsize=18,
             fontweight='bold', y=1.02)
gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)

# Bar chart
ax1 = fig.add_subplot(gs[0])
bars = ax1.bar(CLASS_NAMES, [n_ai, n_real],
               color=[AI_COL, REAL_COL], width=0.5, edgecolor=BG, linewidth=1.5)
for bar, val in zip(bars, [n_ai, n_real]):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + n_total*0.005,
             f'{val:,}', ha='center', color=TEXT, fontweight='bold', fontsize=11)
ax1.set_title('Class Distribution', fontweight='bold', pad=12)
ax1.set_ylabel('Image Count')

# Pie chart
ax2 = fig.add_subplot(gs[1])
wedges, texts, autotexts = ax2.pie(
    [n_ai, n_real], labels=CLASS_NAMES, autopct='%1.1f%%',
    colors=[AI_COL, REAL_COL], startangle=140,
    wedgeprops=dict(edgecolor=BG, linewidth=2.5),
    textprops=dict(color=TEXT)
)
for at in autotexts: at.set_color(BG); at.set_fontweight('bold')
ax2.set_facecolor(PANEL)
ax2.set_title('Class Ratio', color=TEXT, fontweight='bold', pad=12)

# Split breakdown
ax3 = fig.add_subplot(gs[2])
splits = ['Train\n70%', 'Validation\n15%', 'Test\n15%']
counts = [len(train_paths), len(val_paths), len(test_paths)]
split_cols = [ACCENT, GOLD, PURPLE]
bars3 = ax3.bar(splits, counts, color=split_cols, width=0.5, edgecolor=BG, linewidth=1.5)
for bar, val in zip(bars3, counts):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + n_total*0.005,
             f'{val:,}', ha='center', color=TEXT, fontweight='bold', fontsize=11)
ax3.set_title('Train / Val / Test Split', fontweight='bold', pad=12)
ax3.set_ylabel('Image Count')

apply_dark_style(fig, [ax1, ax3])
save_fig(fig, '01_dataset_overview.png')

# ── 3B: Sample Images ───────────────────────────────────────
def load_preview(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    return img.numpy().astype(np.uint8)

n_show = 6
ai_preview   = [p for p, l in all_samples if l == 0][:n_show]
real_preview = [p for p, l in all_samples if l == 1][:n_show]

fig, axes = plt.subplots(2, n_show, figsize=(18, 7), facecolor=BG)
fig.suptitle('Sample Images', color=TEXT, fontsize=16, fontweight='bold')
for i, path in enumerate(ai_preview):
    try:
        axes[0, i].imshow(load_preview(path))
        axes[0, i].axis('off')
        axes[0, i].set_title('AI-Gen', color=AI_COL, fontsize=9, fontweight='bold')
        for spine in axes[0, i].spines.values():
            spine.set_edgecolor(AI_COL); spine.set_linewidth(2)
    except: axes[0, i].axis('off')
for i, path in enumerate(real_preview):
    try:
        axes[1, i].imshow(load_preview(path))
        axes[1, i].axis('off')
        axes[1, i].set_title('Real', color=REAL_COL, fontsize=9, fontweight='bold')
        for spine in axes[1, i].spines.values():
            spine.set_edgecolor(REAL_COL); spine.set_linewidth(2)
    except: axes[1, i].axis('off')

# Row labels
fig.text(0.01, 0.72, '🤖 AI', color=AI_COL, fontsize=12, fontweight='bold', va='center', rotation=90)
fig.text(0.01, 0.28, '📷 Real', color=REAL_COL, fontsize=12, fontweight='bold', va='center', rotation=90)
plt.tight_layout()
save_fig(fig, '02_sample_images.png')

# ── 3C: Image Property Distributions ────────────────────────
SAMPLE_N = min(300, n_total)
sample_items = random.sample(all_samples, SAMPLE_N)
widths, heights, brightnesses = [], [], []
for path, _ in sample_items:
    try:
        raw = tf.io.read_file(path)
        img = tf.image.decode_image(raw, channels=3, expand_animations=False).numpy()
        heights.append(img.shape[0])
        widths.append(img.shape[1])
        brightnesses.append(img.mean() / 255.0)
    except: pass

fig, axes = plt.subplots(1, 3, figsize=(18, 5), facecolor=BG)
fig.suptitle(f'Image Properties (sample of {SAMPLE_N})', color=TEXT, fontsize=15, fontweight='bold')

prop_data  = [widths, heights, brightnesses]
prop_labels= ['Width (px)', 'Height (px)', 'Mean Normalised Pixel']
prop_titles= ['Width Distribution', 'Height Distribution', 'Brightness Distribution']
prop_cols  = [ACCENT, REAL_COL, GOLD]

for ax, data, xlabel, title, col in zip(axes, prop_data, prop_labels, prop_titles, prop_cols):
    ax.hist(data, bins=30, color=col, edgecolor=BG, linewidth=0.8, alpha=0.9)
    ax.set_title(title, fontweight='bold', pad=10)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Count')

apply_dark_style(fig, list(axes))
save_fig(fig, '03_image_properties.png')

print("\nPre-training graphs complete ✓")

  Saved → classifier_outputs\01_dataset_overview.png
  Saved → classifier_outputs\02_sample_images.png
  Saved → classifier_outputs\03_image_properties.png

Pre-training graphs complete ✓


In [4]:
# ============================================================
# CELL 4 — Data Pipeline
# FIX: No /255 here — each model's preprocess_input handles it
# ============================================================

def parse_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)   # ✅ FIX: cast only, NO /255
    img.set_shape([IMG_SIZE[0], IMG_SIZE[1], 3])
    return img, label


def augment(img, label):
    img = tf.image.random_flip_left_right(img)
    # Augment in [0,255] space before model preprocessing
    img = tf.image.random_brightness(img, max_delta=25.5)    # ~10% of 255
    img = tf.image.random_contrast(img, lower=0.85, upper=1.15)
    img = tf.clip_by_value(img, 0.0, 255.0)
    return img, label


def build_dataset(paths, labels, augment_data=False, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(paths), 2000), seed=SEED)
    ds = ds.map(parse_image, num_parallel_calls=AUTOTUNE)
    ds = ds.apply(tf.data.experimental.ignore_errors())  # skip remaining bad images
    if augment_data:
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds


train_ds = build_dataset(train_paths, train_labels, augment_data=True, shuffle=True)
val_ds   = build_dataset(val_paths,   val_labels)
test_ds  = build_dataset(test_paths,  test_labels)

print("Datasets ready ✓")
train_batches = max(1, len(train_paths) // BATCH_SIZE)
val_batches   = max(1, len(val_paths)   // BATCH_SIZE)
test_batches  = max(1, len(test_paths)  // BATCH_SIZE)
print(f"Train batches : {train_batches}")
print(f"Val batches   : {val_batches}")
print(f"Test batches  : {test_batches}")

Instructions for updating:
Use `tf.data.Dataset.ignore_errors` instead.
Datasets ready ✓
Train batches : 267
Val batches   : 57
Test batches  : 57


In [5]:
# ============================================================
# CELL 5 — Model Definitions
# FIX: preprocess_input is inside each model (correct),
#      and fine-tuning only unfreezes last 30 layers (not all)
# ============================================================
from tensorflow.keras import layers, applications
from tensorflow import keras
import numpy as np


def build_head(x):
    """Shared classification head."""
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    return out


# ── MobileNetV2 ──────────────────────────────────────────────
def build_mobilenetv2():
    base = applications.MobileNetV2(
        input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')
    base.trainable = False
    inp = keras.Input(shape=IMG_SIZE + (3,))
    x   = applications.mobilenet_v2.preprocess_input(inp)  # [-1,1]
    x   = base(x, training=False)
    out = build_head(x)
    return keras.Model(inp, out, name='MobileNetV2'), base


# ── EfficientNetB0 ───────────────────────────────────────────
def build_efficientnetb0():
    base = applications.EfficientNetB0(
        input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')
    base.trainable = False
    inp = keras.Input(shape=IMG_SIZE + (3,))
    x   = applications.efficientnet.preprocess_input(inp)   # [0,255] → normalised internally
    x   = base(x, training=False)
    out = build_head(x)
    return keras.Model(inp, out, name='EfficientNetB0'), base


# ── NASNetMobile ─────────────────────────────────────────────
def build_nasnetmobile():
    base = applications.NASNetMobile(
        input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')
    base.trainable = False
    inp = keras.Input(shape=IMG_SIZE + (3,))
    x   = applications.nasnet.preprocess_input(inp)         # [-1,1]
    x   = base(x, training=False)
    out = build_head(x)
    return keras.Model(inp, out, name='NASNetMobile'), base


MODELS = {
    'MobileNetV2'    : build_mobilenetv2,
    'EfficientNetB0' : build_efficientnetb0,
    'NASNetMobile'   : build_nasnetmobile,
}

print("Model builders defined ✓")
for name, builder in MODELS.items():
    m, _ = builder()
    trainable = sum(np.prod(v.shape) for v in m.trainable_weights)
    total     = sum(np.prod(v.shape) for v in m.weights)
    print(f"  {name:<18} | Trainable: {trainable:>10,} | Total: {total:>12,}")
    del m

Model builders defined ✓
  MobileNetV2        | Trainable:    347,009 | Total:    2,607,553
  EfficientNetB0     | Trainable:    347,009 | Total:  4,399,140.0
  NASNetMobile       | Trainable:    289,217 | Total:    4,561,045


In [6]:
# ============================================================
# CELL 6 — Training All Three Models
# FIX 1: Only unfreeze last 30 layers in Phase 2
# FIX 2: Fresh EarlyStopping callback per phase
# ============================================================
histories      = {}
trained_models = {}
trained_bases  = {}
train_times    = {}

LOSS    = 'binary_crossentropy'
METRICS = ['accuracy']

for name, builder in MODELS.items():
    print(f"\n{'='*55}")
    print(f"  Training  →  {name}")
    print(f"{'='*55}")

    model, base = builder()
    t0 = time.time()

    # ── PHASE 1: Train head only ──────────────────────────────
    print("\n🔹 Phase 1: Training classification head...")
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss=LOSS, metrics=METRICS
    )

    # ✅ FIX: fresh callbacks for each phase
    cb_p1 = [
        keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=3,
            restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5,
            patience=2, min_lr=1e-6, verbose=1),
    ]

    history_p1 = model.fit(
        train_ds, validation_data=val_ds,
        epochs=10, callbacks=cb_p1, verbose=1
    )

    # ── PHASE 2: Fine-tune last 30 layers only ────────────────
    print("\n🔹 Phase 2: Fine-tuning last 30 layers of base...")

    # ✅ FIX: unfreeze only last 30 layers, not the full base
    for layer in base.layers:
        layer.trainable = False
    for layer in base.layers[-30:]:
        layer.trainable = True

    model.compile(
        optimizer=keras.optimizers.Adam(1e-5),   # very low LR for fine-tuning
        loss=LOSS, metrics=METRICS
    )

    # ✅ FIX: brand new EarlyStopping — old one's state is stale
    cb_p2 = [
        keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=4,
            restore_best_weights=True, verbose=1),
    ]

    history_p2 = model.fit(
        train_ds, validation_data=val_ds,
        epochs=10, callbacks=cb_p2, verbose=1
    )

    elapsed = time.time() - t0

    # Combine histories
    full_history = {}
    for key in history_p1.history:
        full_history[key] = (
            history_p1.history[key] +
            history_p2.history.get(key, [])
        )

    histories[name]       = full_history
    trained_models[name]  = model
    trained_bases[name]   = base
    train_times[name]     = elapsed

    save_name = name.lower().replace('net', 'net_') + '.keras'
    model.save(str(OUTPUT_DIR / save_name))
    print(f"\n✔ Saved → {OUTPUT_DIR / save_name}")
    print(f"⏱ Total: {elapsed/60:.1f} min")

print("\n✅ All models trained!")


  Training  →  MobileNetV2

🔹 Phase 1: Training classification head...
Epoch 1/10
268/268 ━━━━━━━━━━━━━━━━━━━━ 317s 1s/step - accuracy: 0.7693 - loss: 0.4970 - val_accuracy: 0.8308 - val_loss: 0.3951 - learning_rate: 0.0010
Epoch 2/10
268/268 ━━━━━━━━━━━━━━━━━━━━ 311s 1s/step - accuracy: 0.8347 - loss: 0.3733 - val_accuracy: 0.8357 - val_loss: 0.3628 - learning_rate: 0.0010
Epoch 3/10
268/268 ━━━━━━━━━━━━━━━━━━━━ 310s 1s/step - accuracy: 0.8575 - loss: 0.3271 - val_accuracy: 0.8373 - val_loss: 0.3568 - learning_rate: 0.0010
Epoch 4/10
268/268 ━━━━━━━━━━━━━━━━━━━━ 312s 1s/step - accuracy: 0.8761 - loss: 0.2934 - val_accuracy: 0.8428 - val_loss: 0.3566 - learning_rate: 0.0010
Epoch 5/10
268/268 ━━━━━━━━━━━━━━━━━━━━ 312s 1s/step - accuracy: 0.8872 - loss: 0.2623 - val_accuracy: 0.8455 - val_loss: 0.3531 - learning_rate: 0.0010
Epoch 6/10
268/268 ━━━━━━━━━━━━━━━━━━━━ 309s 1s/step - accuracy: 0.9026 - loss: 0.2385 - val_accuracy: 0.8493 - val_loss: 0.3554 - learning_rate: 0.0010
Epoch 7/10

In [7]:
# ============================================================
# CELL 7 — Post-Training Graphs (saved to OUTPUT_DIR)
# ============================================================

MODEL_COLORS = {
    'MobileNetV2'    : (AI_COL,  '#FF9999'),
    'EfficientNetB0' : (ACCENT,  '#82CFFF'),
    'NASNetMobile'   : (REAL_COL,'#85E89D'),
}

# ── 7A: Training Curves ──────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(20, 11), facecolor=BG)
fig.suptitle('Training History — All Models', color=TEXT,
             fontsize=18, fontweight='bold', y=1.01)

for col, (name, hist) in enumerate(histories.items()):
    c_train, c_val = MODEL_COLORS[name]
    n_epochs = range(1, len(hist['accuracy']) + 1)
    p1_len   = len(histories[name]['accuracy'])   # track phase boundary

    for row, (metric, ylabel) in enumerate([('accuracy', 'Accuracy'), ('loss', 'Loss')]):
        ax = axes[row, col]
        train_vals = hist[metric]
        val_vals   = hist[f'val_{metric}']

        ax.plot(n_epochs, train_vals, color=c_train, lw=2.5, marker='o',
                markersize=4, label='Train', zorder=3)
        ax.plot(n_epochs, val_vals, color=c_val, lw=2.5, marker='s',
                markersize=4, ls='--', label='Validation', zorder=3)

        # Phase separator
        p1_end = len(histories[name]['accuracy']) // 2
        ax.axvline(x=p1_end + 0.5, color=GOLD, lw=1.5, ls=':', alpha=0.8,
                   label='Phase 2 starts')
        ax.text(p1_end + 0.7, ax.get_ylim()[0] if metric=='loss' else 0.02,
                'Fine-tune →', color=GOLD, fontsize=7, alpha=0.85)

        best_val = (max if metric == 'accuracy' else min)(val_vals)
        ax.set_title(f'{name}\n{ylabel}', fontweight='bold', color=TEXT, pad=10)
        ax.set_xlabel('Epoch')
        ax.set_ylabel(ylabel)
        if metric == 'accuracy':
            ax.set_ylim(0, 1.08)
        legend = ax.legend(facecolor=PANEL, edgecolor=BORDER,
                           labelcolor=TEXT, fontsize=8)
        ax.annotate(f"Best Val: {best_val:.3f}",
                    xy=(0.04, 0.90 if metric=='accuracy' else 0.94),
                    xycoords='axes fraction', fontsize=9, color=GOLD,
                    bbox=dict(boxstyle='round,pad=0.4', fc=PANEL,
                              ec=BORDER, alpha=0.9))

apply_dark_style(fig, [ax for row in axes for ax in row])
plt.tight_layout()
save_fig(fig, '04_training_curves.png')

# ── 7B: Training Time ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5), facecolor=BG)
names_  = list(train_times.keys())
times_  = [train_times[n] / 60 for n in names_]
cols_   = [MODEL_COLORS[n][0] for n in names_]
bars_   = ax.barh(names_, times_, color=cols_, edgecolor=BG, linewidth=1.5, height=0.5)
for bar, t in zip(bars_, times_):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{t:.1f} min', va='center', color=TEXT, fontweight='bold')
ax.set_title('Training Time Comparison', fontweight='bold', color=TEXT, pad=12)
ax.set_xlabel('Minutes')
ax.invert_yaxis()
apply_dark_style(fig, [ax])
save_fig(fig, '05_training_time.png')

print("Post-training graphs saved ✓")

  Saved → classifier_outputs\04_training_curves.png
  Saved → classifier_outputs\05_training_time.png
Post-training graphs saved ✓


In [8]:
# ============================================================
# CELL 8 — Evaluation + Confusion Matrices
# ============================================================
from sklearn.metrics import classification_report, confusion_matrix

results_summary = []
all_reports     = {}

for name, model in trained_models.items():
    print(f"\n{'─'*50}")
    print(f"  Evaluating  →  {name}")
    print(f"{'─'*50}")

    loss, acc = model.evaluate(test_ds, verbose=0)
    print(f"  Test Loss     : {loss:.4f}")
    print(f"  Test Accuracy : {acc:.4f}  ({acc*100:.1f}%)")

    y_pred_raw = model.predict(test_ds, verbose=0)
    y_pred     = (y_pred_raw > 0.5).astype(int).flatten()
    y_true     = np.array(test_labels[:len(y_pred)])

    min_len = min(len(y_pred), len(y_true))
    y_pred, y_true = y_pred[:min_len], y_true[:min_len]

    report = classification_report(y_true, y_pred,
                                   target_names=CLASS_NAMES,
                                   zero_division=0, output_dict=True)
    print(classification_report(y_true, y_pred,
                                target_names=CLASS_NAMES, zero_division=0))

    results_summary.append((name, acc, loss))
    all_reports[name] = (y_true, y_pred, report)

# ── Confusion Matrices (all in one figure) ───────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5), facecolor=BG)
fig.suptitle('Confusion Matrices — All Models', color=TEXT,
             fontsize=16, fontweight='bold')

cm_cmaps = ['Reds', 'Blues', 'Greens']
for ax, (name, (y_true, y_pred, _)), cmap in zip(axes, all_reports.items(), cm_cmaps):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                ax=ax, linewidths=1, linecolor=BG,
                annot_kws={'size': 14, 'weight': 'bold', 'color': TEXT})
    ax.set_title(f'{name}', fontweight='bold', color=TEXT, pad=12)
    ax.set_xlabel('Predicted', color=MUTED)
    ax.set_ylabel('Actual', color=MUTED)
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=MUTED)

plt.tight_layout()
save_fig(fig, '06_confusion_matrices.png')

# ── Model Comparison ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor=BG)
fig.suptitle('Model Comparison on Test Set', color=TEXT,
             fontsize=16, fontweight='bold')

names_s = [r[0] for r in results_summary]
accs_s  = [r[1]*100 for r in results_summary]
losses_s= [r[2] for r in results_summary]
cols_s  = [MODEL_COLORS[n][0] for n in names_s]

for ax, vals, ylabel, title in [
    (axes[0], accs_s,   'Accuracy (%)',        'Test Accuracy'),
    (axes[1], losses_s, 'Binary Cross-Entropy', 'Test Loss'),
]:
    bars_ = ax.bar(names_s, vals, color=cols_s, width=0.45,
                   edgecolor=BG, linewidth=1.5)
    for bar, val in zip(bars_, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + max(vals)*0.015,
                f'{val:.2f}', ha='center', color=TEXT,
                fontweight='bold', fontsize=12)
    ax.set_title(title, fontweight='bold', pad=12)
    ax.set_ylabel(ylabel)

apply_dark_style(fig, list(axes))
save_fig(fig, '07_model_comparison.png')

# ── Summary ──────────────────────────────────────────────────
print("\n" + "="*50)
print("  Model Comparison Summary")
print("="*50)
print(f"  {'Model':<18} {'Accuracy':>10} {'Loss':>10}")
print("  " + "-"*40)
for name, acc, loss in sorted(results_summary, key=lambda x: -x[1]):
    print(f"  {name:<18} {acc*100:>9.2f}% {loss:>10.4f}")

best_model_name = max(results_summary, key=lambda x: x[1])[0]
print(f"\n  🏆 Best model: {best_model_name}")


──────────────────────────────────────────────────
  Evaluating  →  MobileNetV2
──────────────────────────────────────────────────
  Test Loss     : 0.3584
  Test Accuracy : 0.8560  (85.6%)
              precision    recall  f1-score   support

AI-Generated       0.88      0.83      0.85       924
        Real       0.83      0.89      0.86       909

    accuracy                           0.86      1833
   macro avg       0.86      0.86      0.86      1833
weighted avg       0.86      0.86      0.86      1833


──────────────────────────────────────────────────
  Evaluating  →  EfficientNetB0
──────────────────────────────────────────────────
  Test Loss     : 0.2983
  Test Accuracy : 0.8729  (87.3%)
              precision    recall  f1-score   support

AI-Generated       0.90      0.84      0.87       924
        Real       0.85      0.90      0.88       909

    accuracy                           0.87      1833
   macro avg       0.87      0.87      0.87      1833
weighted avg    

In [9]:
# ============================================================
# CELL 9 — Per-Class Metrics Chart
# ============================================================
metrics_list = ['precision', 'recall', 'f1-score']
x = np.arange(len(metrics_list))
width = 0.22

for cls in CLASS_NAMES:
    fig, ax = plt.subplots(figsize=(10, 5), facecolor=BG)
    fig.suptitle(f'Per-Model Metrics — {cls}', color=TEXT,
                 fontsize=15, fontweight='bold')

    for i, (name, (_, _, report)) in enumerate(all_reports.items()):
        vals = [report[cls][m] for m in metrics_list]
        offset = (i - 1) * width
        bars_ = ax.bar(x + offset, vals, width, label=name,
                       color=MODEL_COLORS[name][0],
                       edgecolor=BG, linewidth=1)
        for bar, v in zip(bars_, vals):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 0.01,
                    f'{v:.2f}', ha='center', color=TEXT, fontsize=8)

    ax.set_xticks(x)
    ax.set_xticklabels([m.capitalize() for m in metrics_list])
    ax.set_ylim(0, 1.15)
    ax.set_ylabel('Score')
    legend = ax.legend(facecolor=PANEL, edgecolor=BORDER, labelcolor=TEXT)
    apply_dark_style(fig, [ax])
    fname = f"08_metrics_{cls.lower().replace('-','_').replace(' ','_')}.png"
    save_fig(fig, fname)

print("Per-class metric charts saved ✓")

  Saved → classifier_outputs\08_metrics_ai_generated.png
  Saved → classifier_outputs\08_metrics_real.png
Per-class metric charts saved ✓


In [10]:
# ============================================================
# CELL 10 — Prediction Function
# ============================================================

def predict_image(image_path: str,
                  model_name: str = None,
                  confidence_threshold: float = 0.85,
                  show_image: bool = True,
                  save_result: bool = True) -> dict:
    """
    Predict whether a single image is AI-Generated or Real.
    """
    if model_name is None:
        model_name = best_model_name

    model = trained_models[model_name]

    raw  = tf.io.read_file(image_path)
    img  = tf.image.decode_image(raw, channels=3, expand_animations=False)
    img  = tf.image.resize(img, IMG_SIZE)
    img  = tf.cast(img, tf.float32)        # ✅ No /255 — model handles it
    img_batch = tf.expand_dims(img, axis=0)

    raw_score  = float(model.predict(img_batch, verbose=0)[0][0])
    label_idx  = int(raw_score > 0.5)
    label      = CLASS_NAMES[label_idx]
    confidence = raw_score if label_idx == 1 else 1 - raw_score
    status     = 'Certain' if confidence >= confidence_threshold else 'Uncertain'

    if show_image or save_result:
        color = REAL_COL if label == 'Real' else AI_COL
        emoji = '✅' if label == 'Real' else '🤖'
        title = f"{emoji} {label}  ({confidence*100:.1f}%)"
        if status == 'Uncertain': title += '  ⚠️'

        display_img = (img.numpy() / 255.0).clip(0, 1)  # for display only

        fig, axes_ = plt.subplots(1, 2, figsize=(8, 4),
                                  gridspec_kw={'width_ratios': [2, 1]},
                                  facecolor=BG)
        axes_[0].imshow(display_img)
        axes_[0].axis('off')
        axes_[0].set_title(title, color=color, fontweight='bold', fontsize=12)

        # Confidence bar
        axes_[1].barh(['AI-Gen', 'Real'],
                      [1 - raw_score, raw_score],
                      color=[AI_COL, REAL_COL], edgecolor=BG)
        axes_[1].set_xlim(0, 1)
        axes_[1].set_xlabel('Score', color=MUTED)
        axes_[1].set_title('Confidence', color=TEXT, fontsize=10)
        axes_[1].axvline(0.5, color=GOLD, lw=1.5, ls='--', alpha=0.8)
        apply_dark_style(fig, [axes_[1]])

        plt.tight_layout()
        if save_result:
            fname = f"pred_{Path(image_path).stem}.png"
            save_fig(fig, fname)
        elif show_image:
            plt.show()
            plt.close(fig)

    result = {
        'label'      : label,
        'confidence' : round(confidence, 4),
        'raw_score'  : round(raw_score, 4),
        'status'     : status,
        'model_used' : model_name,
    }
    print(f"  Prediction  : {result['label']}")
    print(f"  Confidence  : {result['confidence']*100:.2f}%")
    print(f"  Raw Score   : {result['raw_score']:.4f}")
    print(f"  Status      : {result['status']}")
    print(f"  Model       : {result['model_used']}")
    return result

print("predict_image() ready ✓")
print(f"Default model : {best_model_name}")

# ── Demo on a few test images ─────────────────────────────────
print("\n" + "─"*50)
print(" Demo Predictions on Test Images")
print("─"*50)
demo_paths = random.sample(test_paths[:50], min(4, len(test_paths)))
for path in demo_paths:
    print(f"\n  Image: {Path(path).name}")
    predict_image(path, show_image=False, save_result=True)

predict_image() ready ✓
Default model : EfficientNetB0

──────────────────────────────────────────────────
 Demo Predictions on Test Images
──────────────────────────────────────────────────

  Image: 11896.jpg
  Saved → classifier_outputs\pred_11896.png
  Prediction  : AI-Generated
  Confidence  : 99.91%
  Raw Score   : 0.0009
  Status      : Certain
  Model       : EfficientNetB0

  Image: 13008.jpg
  Saved → classifier_outputs\pred_13008.png
  Prediction  : Real
  Confidence  : 95.54%
  Raw Score   : 0.9554
  Status      : Certain
  Model       : EfficientNetB0

  Image: 21508.jpg
  Saved → classifier_outputs\pred_21508.png
  Prediction  : Real
  Confidence  : 100.00%
  Raw Score   : 1.0000
  Status      : Certain
  Model       : EfficientNetB0

  Image: 10983.jpg
  Saved → classifier_outputs\pred_10983.png
  Prediction  : AI-Generated
  Confidence  : 66.56%
  Raw Score   : 0.3344
  Status      : Uncertain
  Model       : EfficientNetB0


In [11]:
# ============================================================
# CELL 11 — Save Best Model + Final Summary
# ============================================================

# Save all models into OUTPUT_DIR
model_files = {
    'MobileNetV2'    : 'mobilenetv2.keras',
    'EfficientNetB0' : 'efficientnetb0.keras',
    'NASNetMobile'   : 'nasnetmobile.keras',
}
for name, fname in model_files.items():
    trained_models[name].save(str(OUTPUT_DIR / fname))
    print(f"  Saved → {OUTPUT_DIR / fname}")

# Save best model alias
trained_models[best_model_name].save(str(OUTPUT_DIR / 'best_model.keras'))
print(f"  Saved → {OUTPUT_DIR / 'best_model.keras'}  ← {best_model_name}")

print("\n" + "="*60)
print("  PROJECT COMPLETE — Summary")
print("="*60)
for name, acc, loss in sorted(results_summary, key=lambda x: -x[1]):
    tt = train_times[name]
    print(f"  {name:<18}  Acc: {acc*100:.1f}%  Loss: {loss:.4f}  Time: {tt/60:.1f} min")
print(f"\n  🏆 Best: {best_model_name}")
print(f"\n  All outputs saved to: {OUTPUT_DIR.resolve()}")

# List all saved files
print("\n  Saved files:")
for f in sorted(OUTPUT_DIR.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f"    {f.name:<40} {size_kb:>8.1f} KB")

  Saved → classifier_outputs\mobilenetv2.keras
  Saved → classifier_outputs\efficientnetb0.keras
  Saved → classifier_outputs\nasnetmobile.keras
  Saved → classifier_outputs\best_model.keras  ← EfficientNetB0

  PROJECT COMPLETE — Summary
  EfficientNetB0      Acc: 87.3%  Loss: 0.2983  Time: 98.9 min
  MobileNetV2         Acc: 85.6%  Loss: 0.3584  Time: 69.8 min
  NASNetMobile        Acc: 82.2%  Loss: 0.4090  Time: 71.7 min

  🏆 Best: EfficientNetB0

  All outputs saved to: C:\Users\nikhi\OneDrive\Desktop\Classifier\classifier_outputs

  Saved files:
    01_dataset_overview.png                     125.8 KB
    02_sample_images.png                       3028.8 KB
    03_image_properties.png                      99.0 KB
    04_training_curves.png                      307.0 KB
    05_training_time.png                         40.9 KB
    06_confusion_matrices.png                    75.9 KB
    07_model_comparison.png                      96.0 KB
    08_metrics_ai_generated.png             